# 👥 Notebook 2 – Customer Segmentation
**RetailPulse | RFM + K-Means + DBSCAN | 6 Business Segments**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import mlflow, os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0,'../..') 
plt.style.use('dark_background')
os.environ['MLFLOW_TRACKING_URI']='../../mlruns'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
print('Libraries loaded ✅')

In [ ]:
from src.features.feature_engineering import build_rfm
df = pd.read_parquet('../../data/processed/retail_clean.parquet')
rfm = build_rfm(df)
print(f'RFM shape: {rfm.shape}')
rfm.describe()

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,5))
for ax, col in zip(axes, ['Recency','Frequency','Monetary']):
    cap = rfm[col].quantile(0.95)
    rfm[rfm[col]<=cap][col].hist(ax=ax, bins=40, color='#3b82f6', edgecolor='none')
    ax.set_title(f'{col} Distribution')
plt.tight_layout()
plt.savefig('../../reports/segmentation_rfm_dist.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
from src.models.segmentation import train_kmeans, get_pca_coords, segment_summary

rfm_seg = train_kmeans(rfm, n_clusters=6, model_dir='../../models/')
rfm_seg = get_pca_coords(rfm_seg)
summary = segment_summary(rfm_seg)
print(summary.to_string())
rfm_seg.to_parquet('../../data/processed/rfm_segmented.parquet', index=False)
print('\nSaved rfm_segmented.parquet ✅')

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,6))
colors = ['#3b82f6','#10b981','#f59e0b','#ef4444','#8b5cf6','#ec4899']
for i, seg in enumerate(rfm_seg['KMeans_Segment'].unique()):
    mask = rfm_seg['KMeans_Segment']==seg
    axes[0].scatter(rfm_seg[mask]['PCA_1'], rfm_seg[mask]['PCA_2'],
                    label=seg, alpha=0.6, s=15, color=colors[i%6])
axes[0].set_title('Customer Segments – PCA 2D Projection')
axes[0].legend(fontsize=8)

summary_sorted = summary.sort_values('TotalRevenue',ascending=True)
axes[1].barh(summary_sorted['KMeans_Segment'], summary_sorted['TotalRevenue'], color=colors[:len(summary)])
axes[1].set_title('Revenue by Segment')
axes[1].set_xlabel('Total Revenue (£)')
plt.tight_layout()
plt.savefig('../../reports/segmentation_clusters.png',dpi=150,bbox_inches='tight')
plt.show()